In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.metrics import r2_score

plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
AX_WIDTH, AX_HEIGHT = plt.rcParams['figure.figsize']

# 1. Load data

In [ ]:
output_dir = Path("../../results/Spikein/14_spikein_correlation")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = Path("../../results/Spikein/13_filtered/raw_reads.filtered.tsv")
df = pd.read_csv(data_path, sep="\t", header=[0, 1], index_col=[0, 1, 2, 3])

spike_in_sites = {
    "DY215": {"chr": "I", "coord": 3749394, "strand": "-"},
    "DY217": {"chr": "II", "coord": 3344505, "strand": "-"},
    "DY218": {"chr": "II", "coord": 185161, "strand": "-"},
    "DY339": {"chr": "II", "coord": 1157130, "strand": "-"},
    "DY348": {"chr": "II", "coord": 3065244, "strand": "-"}
}

spike_in_ratio = np.array([1.5/100000, 4/100000, 16/100000, 64/100000, 256/100000, 1024/100000])

# 2. Extract the raw count

In [ ]:
spike_data_list = []
for idx, (strain, info) in enumerate(spike_in_sites.items()):
    try:
        row = df.loc[(info["chr"], info["coord"], info["strand"]), :]
        spike_data_list.append({
            "strain": strain,
            "name": f"Spike-in Insertion {idx+1}",
            "chr": info["chr"],
            "coord": info["coord"],
            "strand": info["strand"],
            "reads": row.values[0]  
        })
        print(f"Found {strain}: {info['chr']}:{info['coord']} ({info['strand']})")
    except KeyError:
        print(f"Not found {strain}: {info['chr']}:{info['coord']} ({info['strand']})")

if spike_data_list:
    print(f"\nFound {len(spike_data_list)} spike-in sites")
    
spike_sites = pd.DataFrame([
    {
        "Chr": s["chr"],
        "Coordinate": s["coord"],
        "Strand": s["strand"],
        "Strain": s["strain"],
        "Name": s["name"],
        **{f"Spikein{i}": s["reads"][i] for i in range(6)}
    }
    for s in spike_data_list
])

spike_sites_sorted = spike_sites.sort_values(["Chr", "Coordinate"]).copy()
spike_sites = spike_sites_sorted.sort_values(["Strain"]).reset_index(drop=True).set_index(["Chr", "Coordinate", "Strand", "Name", "Strain"]).rename_axis("Sample", axis=1).stack().to_frame("Reads")

def assign_ratio_by_order(sub_df, spike_in_ratio):

    read_rank = (sub_df["Reads"].rank()-1).astype(int).to_numpy()
    sub_df["Ratio"] = spike_in_ratio[read_rank]
    min_val = sub_df["Reads"].min()
    sub_df["Reads"] = sub_df["Reads"].where(sub_df["Reads"] == min_val, sub_df["Reads"] - min_val)
    sub_df["Relative_Read_Ratio"] = np.log2(sub_df["Reads"] / sub_df["Reads"].max())
    sub_df["Relative_Dilution_Ratio"] = np.log2(sub_df["Ratio"] / sub_df["Ratio"].max())
    return sub_df

spike_sites = spike_sites.groupby("Strain").apply(assign_ratio_by_order, spike_in_ratio=spike_in_ratio).droplevel(0, axis=0)
spike_sites.to_csv(output_dir / "spike_in_results.tsv", index=True, sep="\t")

In [ ]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig, ax = plt.subplots(1, 1, figsize=(6, 6), sharex=True, sharey=True)

for idx, (strain, strain_df) in enumerate(spike_sites.groupby("Name")):

    X = strain_df["Relative_Dilution_Ratio"]
    Y = strain_df["Relative_Read_Ratio"]
    
    ax.scatter(X, Y, label=f"{strain}", facecolor="none", edgecolor=colors[idx], 
               s=150, lw=1.5, alpha=0.75)


slope, intercept, r_value, p_value, std_err = linregress(
    spike_sites["Relative_Dilution_Ratio"],
    spike_sites["Relative_Read_Ratio"]
)
r2_scores = r_value ** 2

line_x = np.array([-10, 0])
line_y = slope * line_x + intercept
ax.plot(line_x, line_y, color="black", ls="--", alpha=0.7, lw=2.5)

ax.tick_params(axis="both", labelsize=16)
ax.set_xlabel("log$_{2}$(relative dilution ratio)", fontsize=17)
ax.set_ylabel("log$_{2}$(relative read ratio)", fontsize=17)
ax.set_xticks([-10, -8, -6, -4, -2, 0])
ax.set_yticks([-10, -8, -6, -4, -2, 0])
ax.set_xticklabels([-10, -8, -6, -4, -2, 0])
ax.set_yticklabels([-10, -8, -6, -4, -2, 0])

ax.text(0.05, 0.95, f"Slope={slope:.2f}\nPCC={r_value:.2f}\nR$^2$={r2_scores:.2f}", 
        fontsize=16, transform=ax.transAxes, ha="left", va="top")
ax.legend(loc="lower right", fontsize=14)

plt.tight_layout()
plt.savefig(output_dir / "spike_in_results.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()